## Description

This notebook uses heuristic and learned perceptual evaluation metrics to quantify the quality of synthetic image datasets for deepfake detector training.

We are most interested in measuring
1. realism (how indistinguishable a synthetic image is from a genuine image)
2. intra-dataset image diversity (degree of variation in image subject, style, color, texture, complexity, etc.) 

## Set-up environment

First, follow the set up instructions in the subnet repo Readme (make sure you have finished running 'python download_data.py')

We are also using Learned Perceptual Image Patch Similarity (LPIPS) metric, which can be installed with:

pip install lpips

https://github.com/richzhang/PerceptualSimilarity

In [1]:
%%time

# Standard libraries
import os
import random
import time
import logging
from typing import List, Dict, Tuple, Any
import json
import multiprocessing
from multiprocessing import Pool, cpu_count
from pathlib import Path
import datetime

# Third-party libraries
import numpy as np
import PIL
import torch
import lpips
import matplotlib.pyplot as plt


# Bitmind-specific libraries
from bitmind.image_dataset import ImageDataset
from bitmind.constants import DATASET_META
from utils import image_utils

/root/miniconda3/envs/bitmind/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU times: user 4.49 s, sys: 2.26 s, total: 6.74 s
Wall time: 5.63 s


### Configs


In [2]:
%%time
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"

CPU times: user 2.78 ms, sys: 37.1 ms, total: 39.8 ms
Wall time: 324 ms


### Load Real Image Datasets

In [3]:
%%time
print("Loading real datasets")
real_image_datasets = [
    ImageDataset(ds['path'], 'test', ds.get('name', None), ds['create_splits'])
    for ds in DATASET_META['real']
]
real_image_datasets

Loading real datasets
CPU times: user 2.04 s, sys: 5.88 s, total: 7.91 s
Wall time: 24 s


### Create Directory of Real Images

In [10]:
def save_image(args):
    index, image, directory_path = args  # Unpack the tuple
    filename = f"{directory_path}/{index}.jpg"
    if image is not None and not os.path.exists(filename):
        image.save(filename)
        return f"Saved {filename}"
    return f"{filename} already exists or no image available"

def sample_images(directory_path : str, index_range, dataset : ImageDataset, n : int):
    print("Process ID: " + str(os.getpid()))
    num_samples = 0
    for idx in index_range:
        try:
            item = dataset[idx]
            if item['image'] is not None:
                save_image((idx, item['image'], directory_path))
                num_samples += 1
        except IndexError:
            # We reached the end of the dataset
            break
        if num_samples == n: return

def get_sample_list(directory_path : str, dataset: ImageDataset, n: int):
    num_processes = max(1, cpu_count() - 1)
    print('Num processes:', num_processes)
    dataset_length = len(dataset)
    
    # Calculate the number of items each process should handle
    items_per_process = dataset_length // num_processes
    # Create the ranges for each process
    ranges = [(i * items_per_process, min((i + 1) * items_per_process, dataset_length)) for i in range(num_processes)]
    # Initialize a multiprocessing pool
    with multiprocessing.Pool(processes=num_processes) as pool:
        # Each process will get a range of indices to process
        pool.starmap(sample_images, [(directory_path, range(start, end), dataset, n // num_processes) for start, end in ranges])

def create_eval_image_dataset(directory_path, dataset, n_samples):
    # Ensure the directory exists
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        
    get_sample_list(directory_path, dataset, n_samples)

In [11]:
%%time
directory_path = os.getcwd()+'/test_data/eval_large_real_image_dataset_test/'
create_eval_image_dataset(directory_path, real_image_datasets[0], 1000)

Num processes: 15
Process ID: 14841Process ID: 14840Process ID: 14843


Process ID: 14844Process ID: 14845Process ID: 14846


Process ID: 14847Process ID: 14848

Process ID: 14849Process ID: 14851Process ID: 14850


Process ID: 14852
Process ID: 14853Process ID: 14854
Process ID: 14855

CPU times: user 45.1 ms, sys: 240 ms, total: 285 ms
Wall time: 37.8 s


### Generated synthetic image directory

In [12]:
def load_image(path, resize_shape=None):
    """
    Load and optionally resize an image from the specified path using PIL.

    Args:
        path (str): The path to the image file.
        resize_shape (tuple of int, optional): The target size as (width, height) if resizing is desired.

    Returns:
        ndarray: The loaded (and possibly resized) image in RGB format.
    """
    # Load image with PIL
    img = PIL.Image.open(path)

    # Resize the image if a resize shape is provided
    if resize_shape is not None:
        img = img.resize(resize_shape, PIL.Image.LANCZOS)

    # Convert the image to RGB if not already in that mode
    if img.mode != 'RGB':
        img = img.convert('RGB')

    # Convert the image to an array and ensure it's 8-bit per channel
    img = np.array(img).astype('uint8')

    return img
    
def im2tensor(image, imtype=np.uint8, cent=1., factor=255./2.):
    return torch.Tensor((image / factor - cent)
                        [:, :, :, np.newaxis].transpose((3, 2, 0, 1)))

def isImageFile(filepath):
    return filepath.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp', '.gif'))

In [19]:
def compute_image_directory_lpip(directory, output_file, max_images=None, use_gpu=False, all_pairs=False, resize_shape=(256, 256)):
    """
    Calculate and write the perceptual distance metric between images in a specified directory using LPIPS.

    Args:
        directory (str): Path to the directory containing images to compare.
        output_file (str): File path where the LPIP distance will be written.
        max_images (int, optional): Maximum number of images to process. If None, processes all images.
        use_gpu (bool, optional): If True, utilizes GPU to accelerate the computation.
        all_pairs (bool, optional): If True, computes distances for all pairs of images. If False, computes only for sequential pairs.

    Returns:
        None: This function writes the results directly to the output file and does not return any value.
    """
    # Initialize the LPIPS model with a predefined neural network (e.g., AlexNet).
    if use_gpu:
        loss_fn = lpips.LPIPS(net='alex').cuda()
    else: loss_fn = lpips.LPIPS(net='alex')

    # Open the output file for writing results
    with open(directory+output_file, 'w') as f:
        files = sorted(os.listdir(directory))
        if max_images is not None:
            files = files[:max_images]

        dists = []  # List to hold computed distances

        total_comparisons = sum(len(files[i+1:]) if all_pairs else 1 for i in range(len(files)-1))
        completed_comparisons = 0
        start_time = time.time()

        # Process each file in the directory
        for i, file in enumerate(files[:-1]):
            if not isImageFile(os.path.join(directory, file)):
                continue
            img0 = lpips.im2tensor(load_image(os.path.join(directory, file), resize_shape))
            if use_gpu:
                img0 = img0.cuda()  # Transfer tensor to GPU if enabled

            # Determine comparison targets based on 'all_pairs' flag
            if all_pairs:
                files1 = files[i+1:]
            else:
                files1 = [files[i+1]] if i + 1 < len(files) else []

            for file1 in files1:
                if not isImageFile(os.path.join(directory, file1)):
                    continue
                img1 = lpips.im2tensor(load_image(os.path.join(directory, file1), resize_shape))
                if use_gpu:
                    img1 = img1.cuda()

                # Compute the perceptual metric between two images
                dist01 = loss_fn.forward(img0, img1)
                f.write('({}, {}): {:.6f}\n'.format(file, file1, dist01.item()))

                dists.append(dist01.item())

                # Update progress
                completed_comparisons += 1
                percentage_complete = (completed_comparisons / total_comparisons) * 100
                elapsed_time = time.time() - start_time
                estimated_total_time = elapsed_time / completed_comparisons * total_comparisons
                estimated_time_remaining = estimated_total_time - elapsed_time
                eta = datetime.datetime.now() + datetime.timedelta(seconds=estimated_time_remaining)
            
                # Convert seconds to hours, minutes, and seconds
                hours, remainder = divmod(estimated_time_remaining, 3600)
                minutes, seconds = divmod(remainder, 60)

                if completed_comparisons % (total_comparisons // 100) == 0 or completed_comparisons == total_comparisons:
                    print(f"Progress: {percentage_complete:.2f}% - Time Remaining: {int(hours):02}:{int(minutes):02}:{int(seconds):02}")
                    print(f"ETA: {eta.strftime('%Y-%m-%d %H:%M:%S')}")

        # Calculate statistics: average distance and standard error
        try:
            avg_dist = np.mean(dists)
            stderr_dist = np.std(dists) / np.sqrt(len(dists))
            print('Avg: {:.5f} +/- {:.5f}'.format(avg_dist, stderr_dist))
            f.write('Avg: {:.6f} +/- {:.6f}'.format(avg_dist, stderr_dist))
        except Exception as e:
            print('Error:', e)

In [33]:
directory_path

'/root/bitmind-subnet/bitmind/synthetic_image_generation/data/eval_large_real_image_dataset_test/'

In [20]:
%%time
compute_image_directory_lpip(directory_path, 'lpip_metrics', max_images=200, use_gpu=True, all_pairs=True, resize_shape=(256,256))

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /root/miniconda3/envs/bitmind/lib/python3.10/site-packages/lpips/weights/v0.1/alex.pth
Progress: 1.00% - ETA: 2024-07-10 20:35:14
Progress: 2.00% - ETA: 2024-07-10 20:34:25
Progress: 3.00% - ETA: 2024-07-10 20:34:13
Progress: 4.00% - ETA: 2024-07-10 20:34:14
Progress: 5.00% - ETA: 2024-07-10 20:34:07
Progress: 6.00% - ETA: 2024-07-10 20:34:04
Progress: 7.00% - ETA: 2024-07-10 20:34:00
Progress: 8.00% - ETA: 2024-07-10 20:33:55
Progress: 9.00% - ETA: 2024-07-10 20:33:59
Progress: 10.00% - ETA: 2024-07-10 20:33:58
Progress: 11.00% - ETA: 2024-07-10 20:33:50
Progress: 12.00% - ETA: 2024-07-10 20:33:46
Progress: 13.00% - ETA: 2024-07-10 20:33:47
Progress: 14.00% - ETA: 2024-07-10 20:33:43
Progress: 15.00% - ETA: 2024-07-10 20:33:43
Progress: 16.00% - ETA: 2024-07-10 20:33:42
Progress: 17.00% - ETA: 2024-07-10 20:33:35
Progress: 18.00% - ETA: 2024-07-10 20:33:33
Progress: 19.00% - ETA: 2024-07-10 20: